# Intelligent Asset Lifecycle Management
## Model 2 — Remaining Useful Life (RUL) Prediction

This notebook builds the second ML component of the project.

**Objective:** predict the remaining useful life of an industrial machine in operating hours.

Target: `rul_hours`

Architecture:

**Load → Inspect → Clean → Feature Engineering → Train/Test Split → Preprocessing → Regression Models → Compare → Best Model → RUL Prediction**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


## 1. Load Dataset

In [ ]:
# Put your CSV in the same folder as this notebook.
# Change the filename if necessary.

DATA_PATH = "/content/drive/MyDrive/Zenesys/industrial_machine_predictive_maintenance.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


## 2. Inspect the Dataset

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
print("RUL statistics:")
print(df["rul_hours"].describe())

plt.figure(figsize=(8, 5))
plt.hist(df["rul_hours"], bins=40)
plt.xlabel("Remaining Useful Life (hours)")
plt.ylabel("Number of Records")
plt.title("Distribution of Remaining Useful Life")
plt.show()


## 3. Basic Cleaning

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

if "machine_id" in df.columns:
    df = df.drop(columns=["machine_id"])

print("New shape:", df.shape)


## 4. Feature Engineering

We create two condition/stress indicators:

- `temperature_rise` = motor temperature − ambient temperature
- `electrical_load_index` = current × RPM


In [ ]:
if {"temperature_motor", "ambient_temp"}.issubset(df.columns):
    df["temperature_rise"] = (
        df["temperature_motor"] - df["ambient_temp"]
    )

if {"current_phase_avg", "rpm"}.issubset(df.columns):
    df["electrical_load_index"] = (
        df["current_phase_avg"] * df["rpm"]
    )

df.head()


## 5. Select Features and Target

Target:

`rul_hours`

Excluded from the inputs:

- `rul_hours` — target itself
- `failure_within_24h` — future failure information
- `failure_type` — describes the failure outcome
- `estimated_repair_cost` — downstream financial information


In [ ]:
TARGET = "rul_hours"

leakage_columns = [
    TARGET,
    "failure_within_24h",
    "failure_type",
    "estimated_repair_cost"
]

X = df.drop(
    columns=[c for c in leakage_columns if c in df.columns]
)

y = df[TARGET]

valid_target = y.notna()
X = X.loc[valid_target].copy()
y = y.loc[valid_target].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())


## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 7. Preprocessing

In [ ]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created.")


## 8. Define Regression Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=8,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    )
}

for name in models:
    print("-", name)


## 9. Train and Compare Models

Metrics:

- **MAE:** average error in hours — lower is better
- **RMSE:** penalizes large errors — lower is better
- **R²:** explained variance — higher is better


In [ ]:
results = []
trained_models = {}

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE (hours)": mae,
        "RMSE (hours)": rmse,
        "R²": r2
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    by="MAE (hours)",
    ascending=True
).reset_index(drop=True)

results_df


## 10. Select the Best RUL Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best RUL model:", best_model_name)
print("MAE:", results_df.iloc[0]["MAE (hours)"], "hours")
print("RMSE:", results_df.iloc[0]["RMSE (hours)"], "hours")
print("R²:", results_df.iloc[0]["R²"])


## 11. Actual vs Predicted RUL

In [ ]:
best_pred = best_model.predict(X_test)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_pred, alpha=0.5)

minimum = min(y_test.min(), best_pred.min())
maximum = max(y_test.max(), best_pred.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.xlabel("Actual RUL (hours)")
plt.ylabel("Predicted RUL (hours)")
plt.title(f"Actual vs Predicted RUL — {best_model_name}")
plt.show()


## 12. RUL Prediction Errors

In [ ]:
rul_results = X_test.copy()

rul_results["actual_rul_hours"] = y_test.values
rul_results["predicted_rul_hours"] = best_pred

rul_results["prediction_error_hours"] = (
    rul_results["predicted_rul_hours"]
    - rul_results["actual_rul_hours"]
)

rul_results["absolute_error_hours"] = (
    rul_results["prediction_error_hours"].abs()
)

rul_results.head()


## 13. Lifecycle Status

These are simple business rules layered on top of the RUL prediction.

They can later be customized for each machine type.


In [ ]:
def lifecycle_status(rul_hours):

    if rul_hours <= 50:
        return "CRITICAL — IMMEDIATE ACTION"
    elif rul_hours <= 150:
        return "HIGH — PLAN MAINTENANCE"
    elif rul_hours <= 300:
        return "MODERATE — MONITOR"
    else:
        return "HEALTHY — NORMAL OPERATION"


example_rul = 120

print("Predicted RUL:", example_rul, "hours")
print("Status:", lifecycle_status(example_rul))


## 14. Generate a Report for One Asset

In [ ]:
sample = X_test.iloc[[0]]
predicted_rul = best_model.predict(sample)[0]
original_row = df.loc[sample.index[0]]

print("======================================")
print("       ASSET RUL REPORT")
print("======================================")
print("Model:", best_model_name)
print("Predicted RUL:", round(predicted_rul, 2), "hours")
print("Lifecycle Status:", lifecycle_status(predicted_rul))

if "hours_since_maintenance" in original_row:
    print(
        "Hours Since Maintenance:",
        round(original_row["hours_since_maintenance"], 2)
    )


## 15. Predict RUL for All Test Assets

In [ ]:
rul_predictions = X_test.copy()

rul_predictions["actual_rul_hours"] = y_test.values
rul_predictions["predicted_rul_hours"] = best_pred

rul_predictions["absolute_error_hours"] = (
    rul_predictions["predicted_rul_hours"]
    - rul_predictions["actual_rul_hours"]
).abs()

rul_predictions["lifecycle_status"] = [
    lifecycle_status(rul)
    for rul in best_pred
]

rul_predictions.head()


## 16. Find Assets With the Lowest Predicted RUL

In [ ]:
critical_assets = rul_predictions.sort_values(
    by="predicted_rul_hours",
    ascending=True
)

critical_assets.head(20)


## 17. RUL Status Summary

In [ ]:
print(rul_predictions["lifecycle_status"].value_counts())


## 18. Save RUL Predictions

These predictions will later be combined with Model 1 (failure probability) and the asset's repair-cost information.


In [ ]:
OUTPUT_PATH = "asset_rul_predictions.csv"

rul_predictions.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)


# Next Stage — Combine Model 1 + Model 2

Model 1:

**Failure Prediction → Failure Probability**

Model 2:

**RUL Prediction → Remaining Hours**

The next stage is the lifecycle decision engine:

**Failure Probability + RUL + Repair Cost + Maintenance History → Monitor / Maintain / Repair / Upgrade / Replace**
